# Assignment 2 Step 2 — relational graph experiment

這一輪已由我在 CPU 實際跑完。你不必現在重跑。這份 notebook 用於重現與日後前向驗證。

- User MLP: 48.23%；Relational GNN: 48.52%；既有 temporal LightGBM: 50.17%。這些是已看過的歷史週次診斷，不是正式成績。
- `MODE='reproduce'`：缺少模型時才建立 baseline 與 GNN，保留本地 checkpoint。
- `MODE='forward'`：只評估凍結模型。須先恢復自己的 checkpoint，不能在看新週答案後重訓候選再稱為前向測試。
- 原點 10/22 和 10/29（台灣午夜）的七日結果，在 10/29 和 11/5 的資料更新後才完整；此前只輸出 PENDING。
- 不需要 HF_TOKEN。預設 CPU；若改 GPU，設 `DEVICE='cuda:0'`。
- 個別使用者預測、原始圖張量與模型檔保留本地，請勿上傳公開 GitHub。


In [ ]:
import sys, subprocess, json, hashlib, zipfile
from pathlib import Path
from datetime import datetime, timezone
MODE = 'reproduce'  # 'reproduce' or 'forward'
DEVICE = 'cpu'
PRIVATE_CHECKPOINT_ZIP = ''  # Optional local path to your private frozen-model backup.
DATA_SNAPSHOT_ZIP = ''       # Optional exact instructor-data backup for reproduction.
BASE = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
PROJECT = BASE / 'Asg2_Graph_Project'
if not PROJECT.exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/ellatso/NYCU-BigData-Assignment2.git',str(PROJECT)],check=True)
print('Project:',PROJECT)
print('Code commit:',subprocess.check_output(['git','-C',str(PROJECT),'rev-parse','HEAD'],text=True).strip())
assert MODE in ['reproduce','forward']


In [ ]:
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(PROJECT/'requirements_graph.txt')],check=True)
import torch
print('PyTorch:',torch.__version__,'CUDA available:',torch.cuda.is_available())


In [ ]:
def safe_extract(path, destination):
    destination = Path(destination).resolve()
    with zipfile.ZipFile(path) as archive:
        for member in archive.infolist():
            if not (destination/member.filename).resolve().is_relative_to(destination):
                raise ValueError('Unsafe archive path')
        archive.extractall(destination)

if PRIVATE_CHECKPOINT_ZIP:
    safe_extract(PRIVATE_CHECKPOINT_ZIP, PROJECT)
if DATA_SNAPSHOT_ZIP:
    safe_extract(DATA_SNAPSHOT_ZIP, BASE/'Asg2_Preserved_Snapshot')
    DATA = BASE/'Asg2_Preserved_Snapshot'/'assignment2_data'
else:
    # Forward mode clones a fresh copy instead of git-pulling a force-pushed data repo.
    suffix = datetime.now(timezone.utc).strftime('_%Y%m%dT%H%M%S') if MODE=='forward' else ''
    DATA = BASE/('Asg2_Official_Data'+suffix)
    if not DATA.exists():
        subprocess.run(['git','clone','--depth','1','https://github.com/nycubigdata-2026fall/BigData2026-Assignment2.git',str(DATA)],check=True)
print('Data:',DATA)


In [ ]:
OUT = PROJECT/'artifacts_step2'
if MODE=='reproduce':
    if not (PROJECT/'artifacts/model_temporal.pkl').exists():
        subprocess.run([sys.executable,'-m','asg2.run','benchmark','--data',str(DATA),'--out',str(PROJECT/'artifacts')],cwd=PROJECT,check=True)
    implementation = hashlib.sha256((PROJECT/'asg2/graph.py').read_bytes()).hexdigest()
    protocol_path = OUT/'protocol.json'
    reusable = protocol_path.exists() and all((OUT/f'{n}.pt').exists() for n in ['user_mlp','relational_gnn'])
    if reusable:
        previous = json.loads(protocol_path.read_text())
        reusable = previous['implementation_sha256']==implementation and previous['epochs_fixed_before_diagnostic']==8
    if not reusable:
        subprocess.run([sys.executable,'-m','asg2.graph_run','train','--data',str(DATA),'--out',str(OUT),'--device',DEVICE,'--epochs','8'],cwd=PROJECT,check=True)
    else:
        print('Reusing frozen neural checkpoints; no retraining.')
else:
    required=[OUT/'user_mlp.pt',OUT/'relational_gnn.pt',PROJECT/'artifacts/model_temporal.pkl']
    if not all(p.exists() for p in required):
        raise RuntimeError('Restore your private frozen checkpoint backup first. Forward mode will not retrain missing candidates.')


In [ ]:
if MODE=='reproduce':
    subprocess.run([sys.executable,'-m','unittest','discover','-s','tests','-v'],cwd=PROJECT,check=True)
    subprocess.run([sys.executable,str(PROJECT/'scripts/check_graph_asof.py'),'--data',str(DATA),'--out',str(OUT)],cwd=PROJECT,check=True)
subprocess.run([sys.executable,'-m','asg2.graph_run','forward','--data',str(DATA),'--out',str(OUT),'--device',DEVICE],cwd=PROJECT,check=True)
import pandas as pd
print('Neural development diagnostics — not a fresh untouched test')
display(pd.read_csv(OUT/'metrics.csv'))
print('Forward status:',(OUT/'forward_status.json').read_text())


In [ ]:
# Keep these privately for future evaluation. This archive contains model
# weights, thresholds and training provenance; no raw per-user graph tensors.
backup = BASE/'Asg2_Frozen_Checkpoints.zip'
selected = [PROJECT/'artifacts/model_temporal.pkl',PROJECT/'artifacts/data_manifest.json',PROJECT/'artifacts/protocol.json']
selected += [OUT/f'{n}.pt' for n in ['user_mlp','relational_gnn']]
selected += [OUT/'protocol.json',OUT/'data_manifest.json']
with zipfile.ZipFile(backup,'w',zipfile.ZIP_DEFLATED) as archive:
    for path in selected:
        if path.exists(): archive.write(path,path.relative_to(PROJECT))
print('Private checkpoint backup:',backup)
print('Keep this archive or saved Kaggle Output; do not publish it to GitHub.')


## What remains

The official test-user list is scheduled for 10/28. The final prediction needs the final 11/11 data release. No official November CSV has been generated. English slides (teacher template, <=10 pages) and a personally narrated 8–12 minute MP4 still need to be prepared.

Step 2 neural models are experiment candidates. The existing final CSV CLI still uses the Step 1 calibration-selected tree model; it is not silently changed to the GNN. Any final model/ensemble choice must be documented and supported by fresh validation.
